## **delicious_grab_zone_log_clean.csv file upload**

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Delicious Grab Zone").getOrCreate()
#Load cleaned csv from Volume
cleaned_df=spark.read.option("header","true").option("inferschema","true").csv("/Volumes/workspace/delicious_grab_zone_schema/delicious_grab_zone_volume/delicious_grab_zone_log_clean.csv/")

In [0]:
#save it as a delta table
cleaned_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("delicious_grab_zone_schema.delicious_grab_zone_delta_cleaned")



In [0]:
cleaned_df.display()

Customer_id,Delivery_date,Customer_name,Item,Quantity,Price,Address,Mode_of_payment
ORD0001,2025-08-01,Karthik,Eggless Classic Brownie,3,250,E101,Cash
ORD0002,2025-08-04,Victoria,Wheat Loaf,1,60,A301,Cash
ORD0003,2025-08-06,Naveen,Classic Brownie (4pcs),3,220,I116,Cash
ORD0004,2025-08-10,Vasantha,Black Forest 1kg (Victoria),1,850,H310,Cash
ORD0005,2025-08-11,Ravi,Blondie,3,110,B107,Card
ORD0006,2025-08-13,Abejith,Classic Brownie (4pcs),3,220,A311,Cash
ORD0007,2025-08-16,Meena,Mango Mousse Cake (Dinesh),1,1120,J310,Cash
ORD0008,2025-08-17,Vasantha,White Loaf,1,110,A202,Cash
ORD0009,2025-08-18,Vasantha,Black Forest 1kg (Victoria),1,850,F104,Cash
ORD0010,2025-08-20,Arun,Wheat Loaf,3,60,E207,Card


In [0]:
#1. Filtering Premium Orders (prize>500)
df_trans1=cleaned_df.filter(cleaned_df["Price"]>500)
df_trans1.display()



Customer_id,Delivery_date,Customer_name,Item,Quantity,Price,Address,Mode_of_payment
ORD0004,2025-08-10,Vasantha,Black Forest 1kg (Victoria),1,850,H310,Cash
ORD0007,2025-08-16,Meena,Mango Mousse Cake (Dinesh),1,1120,J310,Cash
ORD0009,2025-08-18,Vasantha,Black Forest 1kg (Victoria),1,850,F104,Cash
ORD0015,2025-09-02,Abejith,Oreo Cake (Janani),1,1050,E109,UPI
ORD0026,2025-09-18,Dinesh,Black Forest 1kg (Victoria),1,850,E305,Cash
ORD0035,2025-09-28,Janani,Oreo Cake (Janani),1,1050,F104,Card
ORD0037,2025-10-02,Abejith,White Forest Eggless Cake 2kg (Vasantha),1,1500,E101,Card
ORD0043,2025-10-10,Naveen,Mango Mousse Cake (Dinesh),1,1120,F208,Card
ORD0051,2025-10-23,Abejith,Oreo Cake 1kg Eggless (Deepa),1,1150,H204,Card
ORD0057,2025-11-01,Dinesh,Mango Cake 1kg (Sowmya),1,950,E305,Card


In [0]:
# 2. Add calculated column (total_value = price × quantity) 
df_trans2 = cleaned_df.withColumn( "Total_value", cleaned_df["price"] * cleaned_df["quantity"] )
df_trans2.display()

Customer_id,Delivery_date,Customer_name,Item,Quantity,Price,Address,Mode_of_payment,Total_value
ORD0001,2025-08-01,Karthik,Eggless Classic Brownie,3,250,E101,Cash,750
ORD0002,2025-08-04,Victoria,Wheat Loaf,1,60,A301,Cash,60
ORD0003,2025-08-06,Naveen,Classic Brownie (4pcs),3,220,I116,Cash,660
ORD0004,2025-08-10,Vasantha,Black Forest 1kg (Victoria),1,850,H310,Cash,850
ORD0005,2025-08-11,Ravi,Blondie,3,110,B107,Card,330
ORD0006,2025-08-13,Abejith,Classic Brownie (4pcs),3,220,A311,Cash,660
ORD0007,2025-08-16,Meena,Mango Mousse Cake (Dinesh),1,1120,J310,Cash,1120
ORD0008,2025-08-17,Vasantha,White Loaf,1,110,A202,Cash,110
ORD0009,2025-08-18,Vasantha,Black Forest 1kg (Victoria),1,850,F104,Cash,850
ORD0010,2025-08-20,Arun,Wheat Loaf,3,60,E207,Card,180


In [0]:
# 3. Categorize items (Bread, Brownie, Cake) 
from pyspark.sql.functions import when 
df_trans3 =df_trans2.withColumn( "Category", when(cleaned_df["item"].like("%Loaf%"), "Bread") 
                                             .when(cleaned_df["item"].like("%Brownie%"), "Brownie")
                                             .when(cleaned_df["item"].like("%Cake%"), "Cake") 
                                             .otherwise("Other") )
df_trans3.display()                                                           

Customer_id,Delivery_date,Customer_name,Item,Quantity,Price,Address,Mode_of_payment,Total_value,Category
ORD0001,2025-08-01,Karthik,Eggless Classic Brownie,3,250,E101,Cash,750,Brownie
ORD0002,2025-08-04,Victoria,Wheat Loaf,1,60,A301,Cash,60,Bread
ORD0003,2025-08-06,Naveen,Classic Brownie (4pcs),3,220,I116,Cash,660,Brownie
ORD0004,2025-08-10,Vasantha,Black Forest 1kg (Victoria),1,850,H310,Cash,850,Other
ORD0005,2025-08-11,Ravi,Blondie,3,110,B107,Card,330,Other
ORD0006,2025-08-13,Abejith,Classic Brownie (4pcs),3,220,A311,Cash,660,Brownie
ORD0007,2025-08-16,Meena,Mango Mousse Cake (Dinesh),1,1120,J310,Cash,1120,Cake
ORD0008,2025-08-17,Vasantha,White Loaf,1,110,A202,Cash,110,Bread
ORD0009,2025-08-18,Vasantha,Black Forest 1kg (Victoria),1,850,F104,Cash,850,Other
ORD0010,2025-08-20,Arun,Wheat Loaf,3,60,E207,Card,180,Bread


In [0]:
#4. Apply Discount Logic -10% Discount on Brownies
from pyspark.sql.functions import when, col

df_trans4 = df_trans3.withColumn(
    "Offer_Price",
    when(col("Item").like("%Brownie%"), col("Price") * 0.9)  # 10% off brownies
    .otherwise(col("Price"))
)
df_trans4.display()



Customer_id,Delivery_date,Customer_name,Item,Quantity,Price,Address,Mode_of_payment,Total_value,Category,Offer_Price
ORD0001,2025-08-01,Karthik,Eggless Classic Brownie,3,250,E101,Cash,750,Brownie,225.0
ORD0002,2025-08-04,Victoria,Wheat Loaf,1,60,A301,Cash,60,Bread,60.0
ORD0003,2025-08-06,Naveen,Classic Brownie (4pcs),3,220,I116,Cash,660,Brownie,198.0
ORD0004,2025-08-10,Vasantha,Black Forest 1kg (Victoria),1,850,H310,Cash,850,Other,850.0
ORD0005,2025-08-11,Ravi,Blondie,3,110,B107,Card,330,Other,110.0
ORD0006,2025-08-13,Abejith,Classic Brownie (4pcs),3,220,A311,Cash,660,Brownie,198.0
ORD0007,2025-08-16,Meena,Mango Mousse Cake (Dinesh),1,1120,J310,Cash,1120,Cake,1120.0
ORD0008,2025-08-17,Vasantha,White Loaf,1,110,A202,Cash,110,Bread,110.0
ORD0009,2025-08-18,Vasantha,Black Forest 1kg (Victoria),1,850,F104,Cash,850,Other,850.0
ORD0010,2025-08-20,Arun,Wheat Loaf,3,60,E207,Card,180,Bread,60.0


In [0]:
#Create a flag for bulk orders (quantity ≥ 10)
df_trans5 = df_trans4.withColumn(
    "Bulk_Order_Flag",
    when(col("Quantity") >= 5, "Yes").otherwise("No")
)
df_trans5.display()

Customer_id,Delivery_date,Customer_name,Item,Quantity,Price,Address,Mode_of_payment,Total_value,Category,Offer_Price,Bulk_Order_Flag
ORD0001,2025-08-01,Karthik,Eggless Classic Brownie,3,250,E101,Cash,750,Brownie,225.0,No
ORD0002,2025-08-04,Victoria,Wheat Loaf,1,60,A301,Cash,60,Bread,60.0,No
ORD0003,2025-08-06,Naveen,Classic Brownie (4pcs),3,220,I116,Cash,660,Brownie,198.0,No
ORD0004,2025-08-10,Vasantha,Black Forest 1kg (Victoria),1,850,H310,Cash,850,Other,850.0,No
ORD0005,2025-08-11,Ravi,Blondie,3,110,B107,Card,330,Other,110.0,No
ORD0006,2025-08-13,Abejith,Classic Brownie (4pcs),3,220,A311,Cash,660,Brownie,198.0,No
ORD0007,2025-08-16,Meena,Mango Mousse Cake (Dinesh),1,1120,J310,Cash,1120,Cake,1120.0,No
ORD0008,2025-08-17,Vasantha,White Loaf,1,110,A202,Cash,110,Bread,110.0,No
ORD0009,2025-08-18,Vasantha,Black Forest 1kg (Victoria),1,850,F104,Cash,850,Other,850.0,No
ORD0010,2025-08-20,Arun,Wheat Loaf,3,60,E207,Card,180,Bread,60.0,No


In [0]:
#Save Transformed data in Delta table 
df_trans5.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("delicious_grab_zone_schema.delicious_grab_zone_delta_transformed")
